# 01. 部署模式

Agent 有三种典型部署模式，取决于任务时长和交互需求：

| 模式 | 适用场景 | 协议 | 超时 |
|------|----------|------|------|
| Sync | 快速问答（<30s） | HTTP request-response | 30-60s |
| Streaming | 实时交互 | SSE (Server-Sent Events) | 长连接 |
| Background | 长时间任务（分钟-小时） | submit/poll/result | 无限制 |

> **检查点**：能说出三种模式各适用于什么 Agent 场景。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 三种部署模式模拟
from dataclasses import dataclass; import time, uuid

class AgentService:
    def process(self, task: str) -> str:
        return f"Result for '{task}': analyzed successfully."

# Mode 1: Sync
class SyncMode:
    def __init__(self):
        self.agent = AgentService()
    def handle(self, task: str) -> dict:
        result = self.agent.process(task)
        return {"status":"ok","result":result}

# Mode 2: Streaming (SSE)
class StreamingMode:
    def handle(self, task: str):
        steps = ["Analyzing...","Processing...","Done!"]
        for i, step in enumerate(steps):
            yield f"data: {{'step':{i+1},'msg':'{step}'}}\n\n"
            time.sleep(0.1)  # simulate async

# Mode 3: Background
class BackgroundMode:
    def __init__(self):
        self.tasks = {}
    def submit(self, task: str) -> str:
        tid = str(uuid.uuid4())[:8]
        self.tasks[tid] = {"status":"pending","task":task}
        return tid
    def poll(self, tid: str) -> dict:
        t = self.tasks.get(tid,{})
        t["status"] = "completed"
        t["result"] = f"Done: {t.get('task','')}"
        return t

# 演示
print("1. Sync:")
print(SyncMode().handle("analyze data"))

print("\n2. Streaming:")
for chunk in StreamingMode().handle("analyze data"):
    print(f"  {chunk.strip()}")

print("\n3. Background:")
bg = BackgroundMode()
tid = bg.submit("train model")
print(f"  Task ID: {tid}")
print(f"  Poll 1: {bg.poll(tid)}")

1. Sync:
{'status': 'ok', 'result': "Result for 'analyze data': analyzed successfully."}

2. Streaming:
  data: {'step':1,'msg':'Analyzing...'}
  data: {'step':2,'msg':'Processing...'}


  data: {'step':3,'msg':'Done!'}



3. Background:
  Task ID: d5d410cd
  Poll 1: {'status': 'completed', 'task': 'train model', 'result': 'Done: train model'}


## HITL (Human-in-the-Loop)

对于不可逆操作，必须插入人工审批：

```
Agent → propose_action() → [APPROVAL GATE] → human approves? → execute()
```

审批门应该显示：
- 执行什么操作
- 影响范围
- 回滚方案
- 紧急程度

> **检查点**：能设计一个 HITL 审批门的 UI 信息结构。

## 真实 API 实验

流式输出实测：SSE 模式的逐 token 到达（`chat(stream=True)`），感受首 token 延迟与后续吞吐。


In [3]:
# 真实 streaming：逐 token 打印 + 计时
import time
t0 = time.time()
first_token_at = None

if client is None:
    print("[toy] streaming 需要 API client")
else:
    resp = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[{"role": "user", "content": "用三句话解释什么是 agent"}],
        max_tokens=150, stream=True,
    )
    chunks = 0
    for chunk in resp:
        delta = chunk.choices[0].delta.content if chunk.choices else None
        if delta:
            if first_token_at is None:
                first_token_at = time.time()
            chunks += 1
            print(delta, end="", flush=True)
    print(f"\n\n首 token 延迟: {(first_token_at or time.time()) - t0:.2f}s | 总时长: {time.time() - t0:.2f}s | chunks: {chunks}")
    print("观察：流式不改变总时长，但把「等待感」从整段移到首 token — 这就是聊天产品必须流式的原因。")


Agent

是

能够

感知

环境

...

机器人

）

还是

物理



首 token 延迟: 2.10s | 总时长: 2.60s | chunks: 47
观察：流式不改变总时长，但把「等待感」从整段移到首 token — 这就是聊天产品必须流式的原因。


## 练习

1. 实现一个 hybrid 模式：前 5 秒用 streaming 展示进度，如果 5 秒未完成，自动切换到 background 模式。
2. 设计 Agent 的状态持久化方案：如果 background task 的进程崩溃重启，如何恢复？
3. 调研 LangGraph 的 checkpointer 机制，与上面的状态持久化方案对比。